# 🎙️ OmniVoice Stable FIXED Colab
### ✔ No freeze
### ✔ No readline hang
### ✔ Stable tunnel fetch


In [ ]:
%cd /content/
!rm -rf omnivoice-colab
!git clone https://github.com/hahyt6644-gif/omnivoice-colab.git
%cd omnivoice-colab

!rm -rf OmniVoice
!git clone https://github.com/k2-fsa/OmniVoice.git

!apt-get update -y
!apt-get install -y ffmpeg

!pip install -q fastapi uvicorn[standard] python-multipart httpx requests aiofiles nest-asyncio pydub soundfile librosa
!pip install -q -r colab.txt

from IPython.display import clear_output
clear_output()
print('✅ INSTALL DONE')

In [ ]:
import subprocess, time, re
import requests

print('🚀 Starting OmniVoice server...')

# Start server
server = subprocess.Popen([
    'python', 'app.py'
])

# Wait for server readiness
for i in range(30):
    try:
        r = requests.get('http://localhost:7860/ui', timeout=2)
        if r.status_code in [200, 405, 404]:
            print('✅ Server ready')
            break
    except:
        time.sleep(2)

print('🌍 Starting tunnel safely...')

tunnel = subprocess.Popen([
    'cloudflared', 'tunnel', '--url', 'http://localhost:7860', '--no-autoupdate'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
start = time.time()

# SAFE LOOP (NO FREEZE)
while time.time() - start < 120:
    line = tunnel.stdout.readline()
    if not line:
        continue

    print(line.strip())

    if 'trycloudflare.com' in line:
        match = re.search(r'https://[-a-zA-Z0-9]+\\.trycloudflare\\.com', line)
        if match:
            url = match.group(0)
            break

if url:
    print('\n========================')
    print('✅ PUBLIC URL READY')
    print(url)
    print('API:', url + '/api/clone')
    print('UI:', url + '/ui')
    print('========================')
else:
    print('❌ Tunnel failed - rerun cell')

In [ ]:
import nest_asyncio
nest_asyncio.apply()
print('stable runtime active')